# Imports

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from datetime import timedelta
import math

# Summarized Results File

In [ ]:
summarized = "./results/summarized/_SUMMARIZED_FILE_NAME_.csv" #Replace the results file path

# Data from CNN experiments

In [ ]:
sort_levels = {
    "class":1,
    "order":2,
    "family":3,
    "genus":4,
    "species":5,
}

In [ ]:
def format(row):
    row["optimizer"] = row["optimizer"][:6]
    row["delta_t"] = row["end_time"] - row["start_time"]
    row["elapsed_time"] = str(timedelta(seconds=math.floor(row["delta_t"])))

    return row

In [ ]:
best_cnn = pd.read_csv(summarized,
    usecols=['start_time', 'end_time', 'level', 'splitter', 'augmentation',
       'batch_size', 'epochs', 'model', 'learning_rate', 'optimizer', 'reserved_memory', 'best_epoch',
       'train_acc_best_epoch', 'test_acc_best_epoch',]
       )

rename = {"test_acc_best_epoch": "test_acc", "train_acc_best_epoch": "train_acc", "learning_rate":"lr"}

best_cnn["sort_level"] = best_cnn.level.map(sort_levels)
best_cnn = best_cnn.sort_values(by=["splitter", "sort_level", "test_acc_best_epoch", "train_acc_best_epoch"], ascending=[True, True, False, False])
best_cnn = best_cnn.apply(format, axis=1).drop(["start_time","end_time", "sort_level"], axis=1).reset_index(drop=True)
best_cnn[['splitter','elapsed_time','level','best_epoch','test_acc_best_epoch','augmentation','batch_size','epochs','model','learning_rate','optimizer','reserved_memory','train_acc_best_epoch']].rename(rename, axis=1)

# Aggregated Metrics

In [ ]:
a = best_cnn.groupby('level').agg({
    'delta_t': ['min', 'max', 'mean'],
    'test_acc_best_epoch': ['min', 'max', 'mean', "median"]
}).reset_index().set_index("level")

a.loc[:,("delta_t", "min")] = a.loc[:,("delta_t", "min")].map(lambda x: str(timedelta(seconds=math.floor(x))))
a.loc[:,("delta_t", "max")] = a.loc[:,("delta_t", "max")].map(lambda x: str(timedelta(seconds=math.floor(x))))
a.loc[:,("delta_t", "mean")] = a.loc[:,("delta_t", "mean")].map(lambda x: str(timedelta(seconds=math.floor(x))))

a.sort_index(key=lambda x: x.map(lambda y: sort_levels[y]))